In [75]:
#Zakładasz konto na pushover.net
#Tworzysz dwa klucze API — User token (ze strony głównej) i Application token (tworząc nową apkę na pushover.net/apps/build)
#Wpisujesz oba tokeny do pliku .env w swoim projekcie
#Instalujesz aplikację Pushover na telefonie

In [76]:
# importujemy biblioteke dotenv a wlasciwie fumkcje load dotenv 
# importujemy klase open ai z api openain
# importujemy os zeby pobrac klucze api z pliku env
# importujemy pdfRedaer z pypdf zeby odczytac plik pdf w pętli 
# 
from dotenv import load_dotenv
from openai import OpenAI
import os 
from pypdf import PdfReader
import json
import gradio as gr

# importujemy też request jest on odpowiedzalny za obsługe http czyli zapytań z serwera 

import requests 

In [77]:

load_dotenv(override=True)
openai = OpenAI()

In [78]:
# pobieramy z env klucze klucz user i nasz token niezbęde do onbłsugi zapytań z serwwera
# ten klucz usera przekauzjemy do zmienenj dzieki temu aplikacja wie do jakiego usera wysłąć powaidomienie 
pushover_user = os.getenv("PUSHOVER_USER")
# ten klucz przekauzjemy do zmiennej, klucz pomaga zweryfikować która aplikacja wysyła powiadomienie
pushover_token = os.getenv("PUSHOVER_TOKEN") 
# ten url  mówi dokąd wysłać nasze zapytanie a odpowiedz trafi dalej dzieki http
pushover_url = "https://api.pushover.net/1/messages.json"


In [79]:
# tworzymy funkcje która  wysyła powiadomienia na telefon jako argument przekazujemy message, który przekażemy przy wywołaniu funkcji

def push(message):
    # weryfikacja czy wiadomsoć wyszła/ wylogoawnie w konsoli argumentu message
    print(f"push : {message}")
    # tworzymy zmienną która wysle dane niezbędne do ibsługi na serwer  wysyłamy klucz usera, kllucz token i wiadoomsc
    payload = {'user': pushover_user, 'token' : pushover_token, 'message' : message}
    # wysyłamy payload na serwer apki pushover metodą request która słuzy do komuiakcji z serwerem, bibliotek akóry pobralismya do wysłania sluzy metoda post
    # jako 1 argument dajemu url, który wymaga od nas api danego serwera apki a jako niezbedne dane wyrzucamy zmienną payload
    requests.post(pushover_url, data=payload)



In [80]:

push('elo typie ')

push : elo typie 


In [81]:

# tworzymy funkcje która przyjmuje 3 parametry email, name i notes name i notes maja ustawione deafault ustawieniea jesli user wysylajacy wiadomsc ich nie wypełni
# funkcja zbiera informacje o osobie ktora wysle nam powiadomoenie

def record_user_details(email, name='not provided',notes = 'not provided'):
    # uruchamiamy funkcje push zadeklarowaną wyżej dodaje do niej przy jej uruchomieniu powyzsze dane z funkcji
    push(f' record send from {name},  wit email {email}, and notes {notes}')
    # zwramay potwierdzenie dla moedlu ai ze record został wysłany z do koncowego usera
    return {'recorded' : 'ok'}

In [82]:

# definiujemy finckje która uruchomi sie jesli model ai nie zna odpowiedzi na pytnia
# jako parametr definiujemy question

def record_unknown_question(question):
    # znowu uruchamiamy funkcje push i wrzucamy do niej nowe dane
    push(f'Recording {question} asked that i couldnnt answer ')
    # zwracamy do funckji info dla modelu ai
    return {'recorded' : 'ok'}

In [83]:

# teraz tworzymy opis w formacje JSON dla narzęedzua / funkcji  record_user_detail zeby model ai mógł ją obsłuzyć zgodnie z dokmetacja api openAI (functions // tool use)
# to narzędzie zapisuje dane kontaktowe 

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:

# teraz tworzymy opis w formacje JSON dla narzęedzua / funkcji  record_unknown_question zeby model ai mógł ją obsłuzyć zgodnie z dokmetacja api openAI (functions // tool use)
# to narzędzie wysyła push gdy model nie moze odpowiedz na zadane pytanie   

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}


In [85]:
# tworzymy liste tools gdzie łączymy oby 2 instukcje opisy JSON w i liste tools wszystko na podstawie dokuemntacji api OpenAI
# dajemy typ, u nas to funkcja, i definiujemy dfincke dodajac nazwę 


tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [86]:
# tworzymy funkcję która jest uruchamiana w funkcji sterującej chat, tam gdzie tworzymy model ai funkcja przyjmuje tool_calls  czyli listę narzędzi  czyli nasze zdefiniowane funcje record_user_detail i record_unknow
# razem z ich json tool_calls to odpowiedz z api Open ai którego z naszych narzędzi uzyje, w funkcji chat albo 1 albo 2 albo 2 naraz

def handle_tools_call(tool_calls):
    #tworzymy pustą listę do której będa trafiac uzyta narzędzia przy danym wywoałniu funkcji w funkcji chat które potem trafia do moeldu ai i api openaci
    results = []
    #tworzymy pętle na naszej liscie narzędzi chata gpt iterujemy w pętli po całej liscie co wywołanie wykonjjać kod na 1 narzedziu z listy
    for tool_call in tool_calls:
        #kazdy tool.call czyli narzędzie opisane w funkcji i json ma parametr name, przekazujemy go do zmiennej zeby chat gpt mógl ,miec ta zmienną zeby potem odpalać warunki w ifie
        tool_name = tool_call.function.name
        # teraz przekazuejmy za pomoca json load które zaczytuje z danego tool_call z funkcji argumenty czyli te argumenty którze przekazlaismy do funkcji i są w spisie json w paramaetrs
        arguments = json.loads(tool_call.function.arguments)
        # wypisyjemy do konsoli, który tool został uzyty
        print(f"Tool called: {tool_name}", flush=True)

        # tworzy if else który odpali dane narzędzie (funkcje w kórym je stworzylismy ) w zaleznosci od tego kórego narzedzia chche uzyc chat gpt w funkcji chat jesli tool name jest takie samo jak nazwa funkcji
        if tool_name == "record_user_details":
            #to do result trafi ta funkcja i odpali siekonkretne narzedzie z przeakanymia rgumentami ** pakuje argumenty w 1 listę
        
            result = record_user_details(**arguments)
        # teraz odpalamy elif 2 przypadek elif to taki if else
        elif tool_name == 'record_unknown_question':
            result = record_unknown_question(**arguments)

            # do result ładaujemy opis dla api open ai // ustalamy role jako "tool" / content czyli wybik funkcji     return {'recorded' : 'ok'} przekazujemy w formacie json z metodą dumps który zamienia 
            # nasza liste na string oraz identyfikataor narzędzia po id
        results.append({'role': 'tool', 'content' : json.dumps(result) , 'tool_call_id': tool_call.id })

    return results
        


In [87]:

# pobieramy pdf oraz txt

reader = PdfReader("linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ed Donner"


In [88]:
# dajemy prompt 

system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [89]:

# tworzymy funkcje chat która przyjmuje 2 argumenty message, meesage usera który trafia przy wywołaniu funkcji i history czyli historia rozmów



def chat(message,history):
    # budujemy liste messages która trafi do modelu ai
    messages = [{'role':'system','content':system_prompt}] + history + [{'role':'user','content':message}]

    # tworzymy flage która zmienia sie z false na true, póki jest false pętla trwa jesli zminiemy ja w dalszej czesci kodu na true petla while sie zakonczy
    done = False 
    #tworzymy pętle while która działa dopóki done nie zmieni sie na false cztli cały kod wykonuje sie w petli tak długo az funckja sterujaca if else nie zamieni done na True

    while not done:


        # tworzymy model ai w którym dodajemy nasze tools wysyłamy wszsytko do api openAI
        response = openai.chat.completions.create(model= 'gpt-4.1-mini', messages=messages, tools=tools)

        # tworzymy zmienną finish_reason która infofmuje nas dlczagochat gpt zakonczył odpowiedz z api mamy 2 opcje jakie zwrócimy albo "tool_calls" czyli api open ai chce uzyc narzędzia albo stop
        # co oznacza ze api juz nie musi uzywac narzędzi
        finish_reason = response.choices[0].finish_reason

        # tworzymy if else if jesli nasz response zwrócił finish reason jako tool_calls czyli chce uzyc narzędzi
        if finish_reason=='tool_calls':
            # wycikągamy wiadomosc z response
            message = response.choices[0].message
            #wyciągmy listę zamóiem api na narzędzia
            tool_calls = message.tool_calls
            # uruchamiamy funkcje hadne tool call która wykonuje, uzywa narzędzi i zwraca wyniki jako argument przekazujemy wyciagniete tool_calls
            result = handle_tools_call(tool_calls)
            #do listy messsages apendujemy zamówienie narzędzi
            messages.append(message)
            # extendujemy wyniki z narzędzi
            messages.extend(result)
            # tutaj petla wraca na góre i dodajemy wyniki z narzedzi i narzedzia i chat albo odpowiada albo chce uzyc kolejnego narzędzia
        else:
            done = True
    return response.choices[0].message.content










In [91]:
history = []

msg1 = 'ile masz lat'

answer1 = chat(msg1,history)

print(answer1)

Tool called: record_unknown_question
push : Recording ile masz lat asked that i couldnnt answer 
I don't share my exact age publicly, but I can tell you I started coding at age 8 and have worked in technology and entrepreneurship for many years, with experience spanning from the mid-1990s until now. If you're interested, I can share more about my background or professional journey!
